# 3 — Eight-episode Stage 1 smoke gate
Uses one idle physical A100, serially. Four new ID and four OOD episodes cover both methods and both delay conditions. The smoke rows are real manifest rows and count toward the 456 new episodes.


In [ ]:
import os, subprocess, csv, signal
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; OUT=Path.home()/"stage1"; GPU=(Path.home()/"stage1_gpu.txt").read_text().strip(); P=Path.home()/"LIBERO-plus"
# Recheck the frozen card immediately before launch.
line=subprocess.run(["nvidia-smi",f"--id={GPU}","--query-gpu=memory.used,utilization.gpu","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); print("GPU preflight",GPU,line)
used,util=[int(x.strip()) for x in line.split(',')]
if used>=500 or util>=5: raise SystemExit("STOP: frozen GPU is no longer idle")
base={"CUDA_VISIBLE_DEVICES":GPU,"MUJOCO_EGL_DEVICE_ID":GPU,"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","PYTHONUNBUFFERED":"1"}
# Each command selects exactly four rows: one task, one seed, both methods, both delays.
jobs=[("id",Path.home()/"venv-stage1-id/bin/python",["--seed","2","--task","spatial_transport"]),("ood",Path.home()/"venv-stage1-ood/bin/python",["--seed","0","--task","spatial_transport","--perturbation","object_layout"]) ]
for scene,py,filters in jobs:
    env=os.environ.copy(); env.update(base)
    if scene=='ood': env['PYTHONPATH']=str(P)
    log=OUT/f"smoke_{scene}.log"
    cmd=[str(py),"-u","-m","async_vla_benchmark.scripts.run_stage1","--config",str(R/"async_vla_benchmark/configs/stage1.yaml"),"--manifest",str(OUT/"stage1_manifest.csv"),"--output-dir",str(OUT),"--scene",scene,"--resume",*filters]
    with open(log,"ab") as fh: subprocess.run(cmd,cwd=R,env=env,stdout=fh,stderr=subprocess.STDOUT,check=True)
    print(scene,log.read_text().splitlines()[-5:])


In [ ]:
eps=list((OUT/"episodes").glob("*.json")); print("new episode artifacts",len(eps),"expected 8")
if len(eps)!=8: raise SystemExit("STOP: smoke count is not exactly 8")
subprocess.run([str(Path.home()/"venv-stage1-ood/bin/python"),"-m","async_vla_benchmark.scripts.validate_stage1","--manifest",str(OUT/"stage1_manifest.csv"),"--output-dir",str(OUT),"--allow-incomplete"],cwd=R,check=True)
print("STOP HERE. Paste both log tails and validator output for review before notebook 4.")
